# COS 380: Natural Language Processing
## Lab 2: From Tokens to Bag-of-Words Vectors

**Submit**
- `Lab02.ipynb`
- `nlp_toolkit/vectorizer.py`

### Goal
In Lab 1, you converted raw text into tokens. In this lab, you will convert those tokens into numerical Bag-of-Words representations.

**BBC article -> Lab 1 preprocessing/tokenization -> shared vocabulary -> word-to-index mapping -> document vector -> document-term matrix**

## Task 1: Load the BBC Corpus (1 point)

Display the dataset shape, column names, first few rows, and number of articles in each category.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
COURSE_ROOT = None

for folder in [HERE] + list(HERE.parents):
    if (folder / "nlp_toolkit").exists():
        COURSE_ROOT = folder
        break

if COURSE_ROOT is None:
    raise FileNotFoundError("Could not find the course folder containing nlp_toolkit.")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

DATA_PATH = COURSE_ROOT / "data" / "bbc_news" / "bbc_text_cls.xls"

# The supplied file has an .xls filename but is CSV-formatted.
bbc = pd.read_csv(DATA_PATH)

In [2]:
print("Shape:", bbc.shape)
print("Columns:", list(bbc.columns))
display(bbc.head())
display(bbc["labels"].value_counts())

Shape: (2225, 2)
Columns: ['text', 'labels']


,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


labels
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64

## Task 2: Preprocess the Provided Working Corpus (2 points)

Everyone will use the same 10 BBC articles: two from each category.

Reuse the preprocessing and tokenization functions you wrote in Lab 1. Do not rewrite them here.

In [3]:
WORKING_INDICES = [
    0, 1,          # business
    510, 511,      # entertainment
    896, 897,      # politics
    1313, 1314,    # sport
    1824, 1825     # tech
]

working = bbc.loc[WORKING_INDICES, ["text", "labels"]].copy()
working = working.reset_index(drop=True)

display(working[["labels"]])


,labels
0,business
1,business
2,entertainment
3,entertainment
4,politics
5,politics
6,sport
7,sport
8,tech
9,tech


In [4]:
from nlp_toolkit.preprocessing import (
    lowercase,
    remove_urls,
    remove_mentions,
    handle_hashtags,
    remove_punctuation,
    normalize_whitespace,
    tokenize,
    remove_stopwords,
    lemmatize_tokens,
)

def process_text(text):
    """Notebook-only helper that coordinates Lab 1 functions."""
    text = lowercase(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = handle_hashtags(text)
    text = remove_punctuation(text)
    text = normalize_whitespace(text)

    tokens = tokenize(text)
    tokens = remove_stopwords(tokens, keep_negation=True)
    tokens = lemmatize_tokens(tokens)
    return tokens

In [5]:
working["tokens"] = working["text"].apply(process_text)

for i in range(3):
    print(f"Document {i} | {working.loc[i, 'labels']}")
    print(working.loc[i, "tokens"][:25])
    print()


Document 0 | business
['ad', 'sale', 'boost', 'time', 'warner', 'profit', 'quarterly', 'profit', 'u', 'medium', 'giant', 'timewarner', 'jump', '76', '113bn', '£600m', 'three', 'month', 'december', '639m', 'yearearlier', 'firm', 'one', 'big', 'investor']

Document 1 | business
['dollar', 'gain', 'greenspan', 'speech', 'dollar', 'hit', 'high', 'level', 'euro', 'almost', 'three', 'month', 'federal', 'reserve', 'head', 'say', 'u', 'trade', 'deficit', 'set', 'stabilise', 'alan', 'greenspan', 'highlight', 'u']

Document 2 | entertainment
['gallery', 'unveils', 'interactive', 'tree', 'christmas', 'tree', 'receive', 'text', 'message', 'unveil', 'london', 'tate', 'britain', 'art', 'gallery', 'spruce', 'antenna', 'receive', 'bluetooth', 'text', 'send', 'visitor', 'tate', 'message', 'unwrap']



## Task 3: Build a Shared Vocabulary (3 points)

Your tokenizer gives you tokens for each individual document. Now build one shared vocabulary across all 10 tokenized documents.

Implement `build_vocabulary(tokenized_documents)` in `nlp_toolkit/vectorizer.py`.

Requirements:
- include each unique token exactly once;
- return the vocabulary in deterministic order;
- do not use `CountVectorizer`.

In [6]:
from nlp_toolkit.vectorizer import build_vocabulary

toy_documents = [
    ["cat", "sat"],
    ["cat", "slept"]
]

toy_vocabulary = build_vocabulary(toy_documents)

print("Toy vocabulary:", toy_vocabulary)
assert toy_vocabulary == ["cat", "sat", "slept"]

print("Tiny vocabulary test passed.")

Toy vocabulary: ['cat', 'sat', 'slept']
Tiny vocabulary test passed.


In [7]:
tokenized_documents = working["tokens"].tolist()

vocabulary = build_vocabulary(tokenized_documents)

print("Vocabulary size:", len(vocabulary))

Vocabulary size: 1085


## Task 4: Map Words to Vector Positions (2 points)

Implement `create_word_to_index(vocabulary)` in `nlp_toolkit/vectorizer.py`.

Each vocabulary term must receive one unique integer position.

In [8]:
from nlp_toolkit.vectorizer import create_word_to_index

word_to_index = create_word_to_index(vocabulary)

for word, index in list(word_to_index.items())[:10]:
    print(f"{word!r} -> {index}")

'ad' -> 0
'sale' -> 1
'boost' -> 2
'time' -> 3
'warner' -> 4
'profit' -> 5
'quarterly' -> 6
'u' -> 7
'medium' -> 8
'giant' -> 9


### Task 4 response

Why must every document use the same word-to-index mapping?

**Your response:**

Every document uses the same shared vocabulary, so the tokenizer must use the same index per word, so that each index always corresponds to the same word. Otherwise, it might think that word 5 is pizza because of one document, but then when it tries word 5 in another document it might be "chair" so it would get confused.

## Task 5: Turn One Document into a Bag-of-Words Vector (4 points)

Implement `vectorize_document(tokens, word_to_index)` in `nlp_toolkit/vectorizer.py`.

The function counts how many times each vocabulary word occurs in one document.

In [9]:
from nlp_toolkit.vectorizer import vectorize_document

toy_mapping = {
    "cat": 0,
    "sat": 1
}

toy_vector = vectorize_document(
    ["cat", "cat", "sat"],
    toy_mapping
)

print("Toy vector:", toy_vector)
assert list(toy_vector) == [2, 1]

print("Tiny vector test passed.")

Toy vector: [2, 1]
Tiny vector test passed.


In [10]:
first_vector = vectorize_document(
    tokenized_documents[0],
    word_to_index
)

print("Vector length:", len(first_vector))
print("Number of nonzero positions:", np.count_nonzero(first_vector))

Vector length: 1085
Number of nonzero positions: 183


## Task 6: Build the Document-Term Matrix (4 points)

You already know how to vectorize one document. Now reuse that function for every document.

Implement `vectorize_corpus(tokenized_documents, word_to_index)` in `nlp_toolkit/vectorizer.py`.

Remember:
- one row = one document;
- one column = one vocabulary term;
- one cell = the count of that term in that document.

In [11]:
from nlp_toolkit.vectorizer import vectorize_corpus

X_manual = vectorize_corpus(
    tokenized_documents,
    word_to_index
)

print("Document-term matrix shape:", X_manual.shape)

Document-term matrix shape: (10, 1085)


### Task 6 response

Why are many entries in a Bag-of-Words document-term matrix zero?

**Your response:**

Many entries are 0 because most documents do not contain every single word in the vocabulary. If only 2 documents out of 100 contain "Tintinnabulation", then 98 documents will have a 0 in the index for tintinnabulation.

## Task 7: Compare with CountVectorizer (2 points)

This is your first use of **scikit-learn** in COS 380. Scikit-learn is a Python library that provides tools for data preparation and machine learning. In this lab, we are using only its Bag-of-Words tool: `CountVectorizer`.

You have already built the main Bag-of-Words operations yourself. `CountVectorizer` packages those same ideas into a reusable object.

### `fit()`
In scikit-learn, **fit** means **learn from the supplied data**.

For `CountVectorizer`, `fit()` examines the documents, learns the vocabulary, and assigns each vocabulary word a column position.

This corresponds to the work you did with:

```python
vocabulary = build_vocabulary(tokenized_documents)
word_to_index = create_word_to_index(vocabulary)
```

### `transform()`
**Transform** means **use what was learned to convert data into a new representation**.

For `CountVectorizer`, `transform()` converts the documents into Bag-of-Words vectors using the vocabulary learned during `fit()`.

This corresponds to:

```python
X_manual = vectorize_corpus(tokenized_documents, word_to_index)
```

### `fit_transform()`
`fit_transform()` is simply a shortcut that performs `fit()` and then `transform()`.

We will show `fit()` and `transform()` separately first so you can see what each step does.

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer expects documents as strings.
# Our documents are already processed token lists, so join each list with spaces.
processed_documents = [
    " ".join(tokens)
    for tokens in tokenized_documents
]

# Create the vectorizer.
# These settings tell CountVectorizer to use our already-processed tokens
# instead of applying a different set of tokenization rules.
sk_vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False
)

# FIT:
# Learn the vocabulary and assign each vocabulary word a column.
sk_vectorizer.fit(processed_documents)

# TRANSFORM:
# Use that learned vocabulary to convert the documents into vectors.
X_sklearn = sk_vectorizer.transform(processed_documents)

### Comparison 1:  Vocabulary size

Your `build_vocabulary()` created a vocabulary. `CountVectorizer.fit()` also created a vocabulary.

Because both are using the same processed tokens, the vocabulary sizes should match.

In [13]:
print("Your vocabulary size:       ", len(vocabulary))
print("CountVectorizer vocabulary:", len(sk_vectorizer.vocabulary_))

assert len(vocabulary) == len(sk_vectorizer.vocabulary_)

Your vocabulary size:        1085
CountVectorizer vocabulary: 1085


### Comparison 2: Matrix shape

Both matrices represent the same 10 documents using the same vocabulary.

Therefore, they should have the same number of rows and columns.

In [14]:
print("Your matrix shape:       ", X_manual.shape)
print("CountVectorizer shape:  ", X_sklearn.shape)

assert X_manual.shape == X_sklearn.shape

Your matrix shape:        (10, 1085)
CountVectorizer shape:   (10, 1085)


### Comparison 3: Word counts

Now compare several vocabulary words in the first document. The counts should also match.

In [15]:
sk_word_to_index = sk_vectorizer.vocabulary_

shared_words = vocabulary[:5]

print("Counts in document 0:")
for word in shared_words:
    manual_count = X_manual[0, word_to_index[word]]
    sklearn_count = X_sklearn[0, sk_word_to_index[word]]

    print(
        f"{word!r}: "
        f"manual={manual_count}, "
        f"CountVectorizer={sklearn_count}"
    )

    assert manual_count == sklearn_count

Counts in document 0:
'ad': manual=1, CountVectorizer=1
'sale': manual=5, CountVectorizer=5
'boost': manual=2, CountVectorizer=2
'time': manual=3, CountVectorizer=3
'warner': manual=4, CountVectorizer=4


### The shortcut

Now that you have seen the two operations separately, scikit-learn also allows:

```python
X_sklearn = sk_vectorizer.fit_transform(processed_documents)
```

`fit_transform()` means **fit first, then transform**.

### Task 7 response

What do the matching vocabulary size, matrix shape, and word counts tell you about the relationship between the Bag-of-Words vectorizer you built and scikit-learn's `CountVectorizer`?

**Your response:**

The matching vocabulary size, matrix shape, and word counts tell me that the functions do the same thing. Our functions show what countvectorizor does behind the scenes. 

## Task 8: What Does Bag of Words Lose? (2 points)

Use your vectorizer on:

- `dog bites man`
- `man bites dog`

Use one shared vocabulary for both sentences.

In [16]:
sentence_documents = [
    ["dog", "bites", "man"],
    ["man", "bites", "dog"]
]

sentence_vocabulary = build_vocabulary(sentence_documents)
sentence_mapping = create_word_to_index(sentence_vocabulary)

sentence_matrix = vectorize_corpus(
    sentence_documents,
    sentence_mapping
)

print("Vocabulary:", sentence_vocabulary)
print("Sentence 1 vector:", sentence_matrix[0])
print("Sentence 2 vector:", sentence_matrix[1])

Vocabulary: ['dog', 'bites', 'man']
Sentence 1 vector: [1 1 1]
Sentence 2 vector: [1 1 1]


### Task 8 response

1. Are the two vectors the same?
2. Do the two sentences necessarily have the same meaning?
3. What information does Bag of Words preserve?
4. What important information does it discard?

**Your response:**

The two vectors are the exact same, even though the sentences have different orders. The two sentences actually have very different meanings, which shows that two count vectorizors that are identical might be representing sentences with totally different meanings depending on word order. Bag of words preserves what exact words appear and how many occurances of each. It discards order.

## Validation

In [17]:
from course_tools.cos380_checks import validate_lab2

validate_lab2(globals())

COS 380 LAB 2 - VISIBLE SELF-CHECK
✓ build_vocabulary found
✓ build_vocabulary returns unique terms in deterministic order
✓ create_word_to_index found
✓ create_word_to_index preserves vocabulary order
✓ vectorize_document found
✓ vectorize_document counts terms in the correct positions
✓ vectorize_document safely ignores tokens outside the vocabulary
✓ vectorize_corpus found
✓ vectorize_corpus creates one row per document and one column per vocabulary term
✓ vectorize_corpus reuses the same word-to-index mapping for every document
--------------------------------------------------------------
Visible checks passed: 6/6
--------------------------------------------------------------
TASK 7 - SCIKIT-LEARN COMPARISON (UNSCORED)
✓ manual and CountVectorizer vocabulary sizes match
✗ Task 7 comparison diagnostic runs
    TypeError('_ok() takes 1 positional argument but 2 were given')


(6, 6)

# Final Check

Before submitting:
- Run the notebook from top to bottom.
- Confirm the tiny vocabulary test passes.
- Confirm the tiny vector test gives `[2, 1]`.
- Confirm the BBC document-term matrix has 10 rows.
- Complete the short written responses.
- Submit `Lab02.ipynb` and `nlp_toolkit/vectorizer.py`.